# Session 4 · The Poetic Machine
*Cultural Machines: An Introduction*
Based on Leif Weatherby, *Language Machines: Cultural AI and the End of Remainder Humanism* (University of Minnesota Press, 2025).

**In this session you will:** look inside the "attention" mechanism that powers modern AI, see how strongly these models are pulled by repetition, rhyme and parallel structure, and put a model to work on poetic forms to find where its skill ends and its clichés begin.

**Time:** about 75 minutes. For faster replies, use *Runtime → Change runtime type → T4 GPU*.

### How to use this notebook
- This is a **Google Colab notebook**: a page that mixes reading with small pieces of code you can run.
- To run a grey code box, click it and press **Shift + Enter** (or click the ▶ button on its left).
- **Run the boxes in order, top to bottom.** If something breaks, go to *Runtime → Restart session* and start again from the top.
- You never have to *write* code. Where you see text inside quotation marks, like `"this"`, you can change the words and run the box again. That is the whole skill.
- Boxes marked **Setup** load the machinery. You can open them if you are curious, but you do not need to read them.

## The big idea

In 1960 the linguist Roman Jakobson described the functions language can perform: conveying information, expressing feeling, getting someone to act, and so on. One of these he called the **poetic function**: language drawing attention to *itself*, to its own sounds, patterns and structures. Rhyme, rhythm, parallelism and repetition are its tools. "I came, I saw, I conquered" works because its three parts echo one another.

The technology behind today's AI is called the **transformer**, and its core component is **attention**: for every word, the model calculates how much to "look at" every other word in the text. The boldest claim here is that this mechanism is a machine for the poetic function. It builds meaning by relating each part of the text to every other part, in the way a poem does.

We can go further. Jakobson thought poetics was a small corner of linguistics. This course flips that: it calls for a **general poetics** that treats patterning and self-reference as the foundation of meaning, with pointing at the world as something that comes later. Today we see what that looks like from the inside.

## Setup

In [ ]:
#@title Setup: load GPT-2 with attention visible (about a minute)
import torch, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2", attn_implementation="eager")
model.eval()

def _attention(text):
    ids = tok(text, return_tensors="pt")
    with torch.no_grad():
        out = model(**ids, output_attentions=True)
    # average over all layers and heads, skipping the very first token
    att = torch.stack(out.attentions)[:, 0].mean(dim=(0, 1)).numpy()
    words = [tok.decode([i]).strip() or "·" for i in ids["input_ids"][0]]
    return att, words

def attention_map(text):
    att, words = _attention(text)
    att, words = att[1:, 1:], words[1:]      # drop the first token, which soaks up attention
    att = att / att.sum(axis=1, keepdims=True)
    plt.figure(figsize=(0.55 * len(words) + 3, 0.5 * len(words) + 2))
    plt.imshow(att, cmap="Purples")
    plt.xticks(range(len(words)), words, rotation=90); plt.yticks(range(len(words)), words)
    plt.xlabel("...is looking at this word"); plt.ylabel("This word...")
    plt.title("Where each word looks (darker = more attention)")
    plt.colorbar(shrink=0.6); plt.tight_layout(); plt.show()

def what_does_it_look_at(text, word_position=-1, top=8):
    att, words = _attention(text)
    row = att[word_position].copy(); row[0] = 0; row = row / row.sum()
    order = np.argsort(row)[::-1][:top]
    print(f"The word {words[word_position]!r} pays most attention to:")
    plt.figure(figsize=(8, 3))
    plt.bar([f"{words[i]} ({i})" for i in order], row[order], color="indigo")
    plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

def next_word_table(text, k=8):
    ids = tok(text, return_tensors="pt")["input_ids"]
    with torch.no_grad():
        probs = torch.softmax(model(ids).logits[0, -1], dim=-1)
    top = torch.topk(probs, k)
    return pd.DataFrame([(tok.decode([int(i)]), f"{float(p)*100:.1f}%")
                         for p, i in zip(top.values, top.indices)],
                        columns=["next token", "probability"])

print("Ready.")

## Part 1 · Seeing attention

The grid below shows, for a line of verse, how much each word attends to each earlier word (a model like this can only look backwards). Read it row by row: each row is one word looking back across the line.

In [ ]:
attention_map("The river remembers the rain, and the rain remembers the sky")

**Look for:** does the second "rain" look back at the first? Does the second "remembers" find its twin? Repetition creates strong links. This is the model noticing the *structure* of the line, not its facts.

Try a line of your own: a proverb, a lyric you wrote, a line from a praise poem (your own words).

In [ ]:
attention_map("Write your own line of verse here and run the box")

## Part 2 · Follow one word

Pick a sentence and see which earlier words the **final** word attends to most.

In [ ]:
what_does_it_look_at("When the drummer stopped, the dancers stopped, and even the children stopped")

## Part 3 · The pull of pattern

If the poetic function is built into the machine, it should be very sensitive to parallelism. Compare the model's next-word guesses with and without a pattern set up in advance.

In [ ]:
print("WITHOUT a pattern:")
display(next_word_table("She sang in the morning and she sang at"))
print("\nWITH a pattern:")
display(next_word_table("Drum of morning, drum of noon, drum of"))

And here is something stranger. Below is a list of random, unrelated words, repeated. There is no meaning to follow at all, only form. Watch how confident the model becomes about repeating the sequence the second time round.

In [ ]:
display(next_word_table("purple ladder cotton whisper engine purple ladder cotton"))

Researchers call this behaviour "induction": once a pattern appears, the model's attention hunts for where it happened before and copies what came next. In Jakobson's language, the model projects *equivalence* across the sequence. It is a mechanical ear for echo.

**Try:** set up a rhyme scheme (`"The cat sat on the mat, the dog lay on the log, the bee sat in the"`) or a call-and-response structure, and see what it predicts.

In [ ]:
next_word_table("The cat sat on the mat, the frog sat on the log, the bee flew to the")

## Part 4 · Forms and their limits

Now a small chat model. We ask it for strict poetic forms and check the results. Where does it hold the form? Where does it slip into the most obvious phrasing?

In [ ]:
#@title Setup: load a small chat model (takes 1–3 minutes the first time)
# A small, openly available chat model. It is far weaker than ChatGPT or Claude,
# which is useful: its habits and defaults are easier to see.
import torch, textwrap
from transformers import AutoTokenizer, AutoModelForCausalLM

CHAT_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
chat_tok = AutoTokenizer.from_pretrained(CHAT_MODEL)
chat_model = AutoModelForCausalLM.from_pretrained(CHAT_MODEL).to(device=device, dtype=dtype)

def ask(prompt, system="You are a helpful assistant.", temperature=0.8,
        max_new_tokens=250, show=True):
    """Send one message to the chat model and return its reply."""
    msgs = [{"role": "system", "content": system},
            {"role": "user", "content": prompt}]
    text = chat_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = chat_tok(text, return_tensors="pt").to(device)
    out = chat_model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True,
                              temperature=temperature, top_p=0.9,
                              pad_token_id=chat_tok.eos_token_id)
    reply = chat_tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if show:
        for para in reply.split("\n"):
            print(textwrap.fill(para, 90) if para.strip() else "")
    return reply

print(f"Chat model ready (running on {device}).")

In [ ]:
poem = ask("Write a haiku about a market at dawn. Only the haiku, nothing else.", max_new_tokens=60)

In [ ]:
poem = ask("Write a four-line poem about a power cut during a concert, "
           "with the rhyme scheme ABAB. Only the poem.", max_new_tokens=120)

print("\n--- last word of each line (check the rhymes) ---")
for line in [l for l in poem.split("\n") if l.strip()]:
    print(line.strip().split()[-1].strip(".,;:!?"))

In [ ]:
# Choose any form: a praise poem, a call-and-response song, a limerick, a ghazal,
# a pantoum, an acrostic of your organisation's name...
ask("Write an acrostic poem on the word ARCHIVE. Only the poem.", max_new_tokens=120)

### Worksheet
1. Did it keep the form (syllables, rhyme, line count)?
2. Circle the phrases you have read a hundred times before. How many are there?
3. Ask the same thing again. How similar is the second attempt to the first?

You will likely find that the model is fluent in form but pulled towards the most familiar images: sunrise, whispers, dancing shadows. This tension, between a machine built for the poetic function and a machine that keeps defaulting to cliché, is exactly where Session 5 begins.

## Discussion

1. The claim is that pattern and self-reference are *more* fundamental to language than pointing at the world. After Part 3, how plausible does that seem?
2. Many oral traditions (griots, praise singers, call-and-response) rely on repetition and parallelism as memory and meaning. What would a scholar of oral literature notice about these machines that an engineer might miss?
3. If a machine can hold a form, what remains the poet's job?

## Glossary
- **Transformer**: the design behind modern language models (2017).
- **Attention**: the mechanism by which each word weighs its relationship to other words.
- **Poetic function**: Jakobson's term for language focusing on its own form.
- **General poetics**: a proposed study of meaning that starts from form and pattern across all systems, including computation.

## Going further
- Roman Jakobson, "Linguistics and Poetics" (1960).